In [148]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

In [149]:
PROCESSED_DATA_DIR = Path("../data/processed")

In [150]:
trainers_course_df = pd.read_csv(PROCESSED_DATA_DIR / "trainers_course.csv", sep=",", encoding="utf-8")

trainers_local_df = pd.read_csv(PROCESSED_DATA_DIR / "trainers_local.csv", sep=",", encoding="utf-8")

info_actions_df = pd.read_csv(PROCESSED_DATA_DIR / "info_actions.csv", sep=",", encoding="utf-8")

In [151]:
workforce_df = trainers_course_df.merge(
    trainers_local_df, how="inner", on="trainer_id", validate="many_to_many"
)

workforce_df = workforce_df.merge(
    info_actions_df, how="inner", on="course_id", validate="many_to_one"
)

workforce_df = workforce_df.loc[workforce_df["active"]].drop_duplicates(subset=["trainer_id", "course_id", "local"])

workforce_df

,trainer_id,course_id,local,course_name,course_area,total_hours,active
0,1,CYB,Centro 4,Cibersegurança Básica,Informática,16,True
1,1,PBI,Centro 4,Power BI e Visualização de Dados,Informática,25,True
2,2,CYB,Centro 2,Cibersegurança Básica,Informática,16,True
3,2,PBI,Centro 2,Power BI e Visualização de Dados,Informática,25,True
4,3,GER,Centro 5,Geriatria e Apoio ao Idoso,Saúde,50,True
...,...,...,...,...,...,...,...
381,179,ING,Centro 5,Inglês Profissional,Línguas,30,True
382,180,VND,Centro 3,Vendas e Negociação,Comercial,16,True
383,181,SOC,Centro 2,Socorrismo Básico,Saúde e Segurança,16,True
384,182,HOT,Centro 4,Housekeeping e Operações Hoteleiras,Turismo,25,True


In [152]:
total_trainers = workforce_df["trainer_id"].nunique()
total_courses = workforce_df["course_id"].nunique()
total_locals = workforce_df["local"].nunique()

print(f'Total trainers: {total_trainers}\nTotal courses: {total_courses}\nTotal locals: {total_locals}')

Total trainers: 183
Total courses: 24
Total locals: 5


In [153]:
course_trainer_count = (
    workforce_df.groupby(["course_id", "course_name"])
    .agg(
        trainer_count=("trainer_id", "nunique")
    )
    .assign(
        diff_mean= lambda df: df["trainer_count"].sub(df["trainer_count"].mean()).round(2)
    )
    .sort_values(by="trainer_count", ascending=False)
    .reset_index()
)

course_trainer_count

,course_id,course_name,trainer_count,diff_mean
0,PBI,Power BI e Visualização de Dados,19,5.04
1,GER,Geriatria e Apoio ao Idoso,17,3.04
2,ING,Inglês Profissional,16,2.04
3,FOR,Formação Pedagógica Inicial de Formadores,16,2.04
4,CYB,Cibersegurança Básica,15,1.04
5,PRO,Gestão de Projetos,15,1.04
6,PRI,Primeiros Socorros,15,1.04
7,HAC,Higiene e Segurança Alimentar,15,1.04
8,EMP,Empilhadores e Movimentação de Cargas,14,0.04
9,LID,Liderança e Comunicação,14,0.04


In [154]:
print(f'Variance: {(course_trainer_count["trainer_count"].var()).round(2)}')

course_trainers_quantiles = (
    course_trainer_count["trainer_count"]
    .quantile(q=[qt / 100 for qt in range(5, 100, 5)])
    .rename("trainer_count")
    .rename_axis("quantile")
    .reset_index()
)

course_trainers_quantiles

Variance: 3.43


,quantile,trainer_count
0,0.05,11.15
1,0.10,12.00
2,0.15,12.45
3,0.20,13.00
4,0.25,13.00
5,0.30,13.00
6,0.35,13.00
7,0.40,13.00
8,0.45,13.35
9,0.50,14.00


In [155]:
local_trainer_count = (
    workforce_df.groupby("local")
    .agg(
        trainer_count=("trainer_id", "nunique")
    )
    .assign(
        pct_trainer= lambda df: df["trainer_count"].div(total_trainers).mul(100).round(2)
    )
    .sort_values(by="trainer_count", ascending=False)
    .reset_index()
)

local_trainer_count

,local,trainer_count,pct_trainer
0,Centro 2,44,24.04
1,Centro 3,44,24.04
2,Centro 5,41,22.40
3,Centro 1,39,21.31
4,Centro 4,39,21.31


In [156]:
area_trainers_count = (
    workforce_df.groupby("course_area")
    .agg(
        trainer_count=("trainer_id", "nunique")
    )
    .assign(
        pct_trainer= lambda df: df["trainer_count"].div(total_trainers).mul(100).round(2)
    )
    .sort_values(by="trainer_count", ascending=False)
    .reset_index()
)

area_trainers_count

,course_area,trainer_count,pct_trainer
0,Segurança,36,19.67
1,Informática,33,18.03
2,Gestão,31,16.94
3,Alimentar,25,13.66
4,Saúde,25,13.66
5,Saúde e Segurança,24,13.11
6,Turismo,18,9.84
7,Formação,16,8.74
8,Línguas,16,8.74
9,Comercial,14,7.65


In [157]:
area_trainers_count["trainer_count"].quantile(q=[0.25, 0.5, 0.75])

0.25    15.5
0.50    21.0
0.75    26.5
Name: trainer_count, dtype: float64

In [158]:
course_local_trainers_count = (
    workforce_df.groupby(["course_id", "course_name", "local"])
    .agg(
        trainer_count=("trainer_id", "nunique")
    )
    .assign(
        pct_trainer= lambda df: df["trainer_count"].div(total_trainers).mul(100).round(2)
    )
    .sort_values(by="trainer_count", ascending=False)
    .reset_index()
)

course_local_trainers_count.head(20)

,course_id,course_name,local,trainer_count,pct_trainer
0,PBI,Power BI e Visualização de Dados,Centro 4,6,3.28
1,HAC,Higiene e Segurança Alimentar,Centro 2,5,2.73
2,HCP,HACCP Aplicado,Centro 1,4,2.19
3,ING,Inglês Profissional,Centro 2,4,2.19
4,SCI,Segurança Contra Incêndios,Centro 3,4,2.19
5,HAC,Higiene e Segurança Alimentar,Centro 5,4,2.19
6,PBI,Power BI e Visualização de Dados,Centro 2,4,2.19
7,HCP,HACCP Aplicado,Centro 2,4,2.19
8,HCP,HACCP Aplicado,Centro 3,4,2.19
9,PYT,Introdução à Programação em Python,Centro 5,4,2.19


In [159]:
print(f'Variance: {(course_local_trainers_count["trainer_count"].var()).round(2)}')

course_local_trainers_quantiles = (
    course_local_trainers_count["trainer_count"]
    .quantile(q=[qt / 100 for qt in range(5, 100, 5)])
    .rename("trainer_count")
    .rename_axis("quantile")
    .reset_index()
)

course_local_trainers_quantiles

Variance: 0.79


,quantile,trainer_count
0,0.05,2.0
1,0.10,2.0
2,0.15,2.0
3,0.20,2.0
4,0.25,2.0
5,0.30,3.0
6,0.35,3.0
7,0.40,3.0
8,0.45,3.0
9,0.50,3.0


In [160]:
trainer_features = (
    workforce_df
    .groupby("trainer_id", as_index=False)
    .agg(
        course_count=("course_id", "nunique"),
        local_count=("local", "nunique"),
        area_count=("course_area", "nunique")
    )
    .assign(
        mobile=lambda df: df["local_count"].gt(1),
        multicourse=lambda df: df["course_count"].gt(1),
        multi_area=lambda df: df["area_count"].gt(1),
        local_exclusive=lambda df: df["local_count"].eq(1)
    )
)

trainer_features

,trainer_id,course_count,local_count,area_count,mobile,multicourse,multi_area,local_exclusive
0,1,2,1,1,False,True,False,True
1,2,2,1,1,False,True,False,True
2,3,2,1,2,False,True,True,True
3,4,3,1,2,False,True,True,True
4,5,2,1,1,False,True,False,True
...,...,...,...,...,...,...,...,...
178,179,1,1,1,False,False,False,True
179,180,1,1,1,False,False,False,True
180,181,1,1,1,False,False,False,True
181,182,1,1,1,False,False,False,True


In [161]:
workforce_enriched = workforce_df.merge(
    trainer_features[
        [
            "trainer_id",
            "course_count",
            "local_count",
            "area_count",
            "mobile",
            "multicourse",
            "multi_area",
            "local_exclusive"
        ]
    ],
    on="trainer_id",
    how="left",
    validate="many_to_one"
)

In [163]:
course_local_workforce = (
    workforce_enriched
    .groupby(
        ["course_id", "course_name", "course_area", "local"],
        as_index=False
    )
    .agg(
        trainer_count=("trainer_id", "nunique"),
        mobile_trainer_count=("mobile", "sum"),
        multicourse_trainer_count=("multicourse", "sum"),
        local_exclusive_trainer_count=("local_exclusive", "sum"),
        avg_courses_per_trainer=("course_count", "mean"),
        avg_locations_per_trainer=("local_count", "mean")
    )
    .assign(
        avg_courses_per_trainer=lambda df:
            df["avg_courses_per_trainer"].round(2),

        avg_locations_per_trainer=lambda df:
            df["avg_locations_per_trainer"].round(2),

        single_trainer_dependency=lambda df:
            df["trainer_count"].eq(1),

        low_redundancy=lambda df:
            df["trainer_count"].le(2),

        backup_trainer_count=lambda df:
            df["trainer_count"].sub(1).clip(lower=0),

        pct_mobile_trainers=lambda df:
            df["mobile_trainer_count"]
            .div(df["trainer_count"])
            .mul(100)
            .round(2),

        pct_local_exclusive_trainers=lambda df:
            df["local_exclusive_trainer_count"]
            .div(df["trainer_count"])
            .mul(100)
            .round(2)
    )
)

course_local_workforce

,course_id,course_name,course_area,local,trainer_count,mobile_trainer_count,multicourse_trainer_count,local_exclusive_trainer_count,avg_courses_per_trainer,avg_locations_per_trainer,single_trainer_dependency,low_redundancy,backup_trainer_count,pct_mobile_trainers,pct_local_exclusive_trainers
0,CYB,Cibersegurança Básica,Informática,Centro 1,3,2,3,1,2.33,1.67,False,False,2,66.67,33.33
1,CYB,Cibersegurança Básica,Informática,Centro 2,3,0,2,3,1.67,1.00,False,False,2,0.00,100.00
2,CYB,Cibersegurança Básica,Informática,Centro 3,4,2,4,2,2.50,1.50,False,False,3,50.00,50.00
3,CYB,Cibersegurança Básica,Informática,Centro 4,4,1,4,3,2.00,1.25,False,False,3,25.00,75.00
4,CYB,Cibersegurança Básica,Informática,Centro 5,4,1,4,3,2.50,1.25,False,False,3,25.00,75.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,VND,Vendas e Negociação,Comercial,Centro 1,4,2,3,2,2.00,1.50,False,False,3,50.00,50.00
116,VND,Vendas e Negociação,Comercial,Centro 2,4,1,4,3,2.25,1.25,False,False,3,25.00,75.00
117,VND,Vendas e Negociação,Comercial,Centro 3,4,1,3,3,2.25,1.25,False,False,3,25.00,75.00
118,VND,Vendas e Negociação,Comercial,Centro 4,2,0,2,2,2.50,1.00,False,True,1,0.00,100.00
